# MTrans on Kaggle

Trains [MTrans](https://github.com/Cliu2/MTrans) (ECCV 2022) on KITTI and writes its 3D boxes as KITTI-format pseudo labels. The training code is the repo's own. This notebook adds what a time-limited Kaggle session needs: a checkpoint after every epoch, automatic resume, and a clean stop before the session limit.

## Before you run

1. **Settings → Accelerator:** GPU T4 x2 (one GPU is used). **Internet:** On.
2. **Add Input:** a KITTI 3D object detection dataset that contains `training/image_2`, `training/velodyne`, `training/calib` and `training/label_2`, for example `garymk/kitti-3d-object-detection-dataset`. The notebook finds these folders wherever they are mounted.
3. For the long run use **Save Version → Save & Run All**. Training stops by itself after `TIME_BUDGET_HOURS`.

## Continuing in a later session

Add the previous version's output of this notebook as an input (**Add Input → Notebook Output**) and run it again. It picks up `last.pt` and the dataset cache from that output and carries on from the next epoch. Keep `EXPERIMENT` and `EPOCHS` unchanged between sessions, because the learning-rate schedule depends on the total number of epochs.

## 1. Settings

In [ ]:
import codecs, json, os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

T0 = time.time()   # TIME_BUDGET_HOURS is measured from here

# ---- what to run ----
RUN_SMOKE_TEST = True    # one tiny epoch + evaluation + label export on 64 objects, to catch errors early
RUN_TRAINING   = True
EXPORT_LABELS  = True    # once the last epoch is done, write KITTI-format labels for LABEL_SPLIT
LABEL_SPLIT    = 'val'   # 'val' or 'train'

# ---- training: the defaults are the repo's configs/MTrans_kitti.yaml ----
EXPERIMENT        = 'mtrans_kitti'
EPOCHS            = 300
EVAL_EVERY        = 5
EVAL_BEGIN        = 250   # no validation before this epoch; lower it to see validation IoU earlier
LABELLED_FRAMES   = 500   # frames whose 3D labels are used; the other training frames are used without labels
NUM_WORKERS       = os.cpu_count() or 2
TIME_BUDGET_HOURS = 11.0  # stop cleanly this long after the notebook started (a Kaggle session ends at 12 h)
USE_COMPILED_IOU  = True  # try the repo's CUDA extension; a pure-PyTorch version is used if it does not build

# ---- where things come from ----
REPO_URL    = 'https://github.com/Cliu2/MTrans.git'
CODE_INPUT  = None        # or a folder under /kaggle/input with your own copy of the repo (train.py at its top level)
RESUME_FROM = None        # or the path of a checkpoint; by default the furthest last.pt under /kaggle/input is used

# ---- paths ----
INPUT_ROOT  = Path('/kaggle/input')
REPO        = Path('/kaggle/working/MTrans')
DATA_ROOT   = Path('/kaggle/temp/kitti_detect')     # scratch: links to the dataset plus per-frame files the repo writes
CACHE_DIR   = Path('/kaggle/working/kitti_cache')   # object-level cache, saved with the output so later sessions skip the build
OUTPUT_ROOT = Path('/kaggle/working/output')
LABEL_DIR   = Path('/kaggle/working/pseudo_label')
CFG_FILE    = REPO / 'configs' / 'MTrans_kaggle.yaml'
CKPT_DIR    = OUTPUT_ROOT / EXPERIMENT / 'ckpt'


def sh(cmd, cwd=None):
    """Run a shell command, stream its output here, and raise if it fails (so Run All stops)."""
    print(f'$ {cmd}', flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    while True:
        chunk = os.read(proc.stdout.fileno(), 4096)
        if not chunk:
            break
        sys.stdout.write(decoder.decode(chunk))
        sys.stdout.flush()
    if proc.wait() != 0:
        raise RuntimeError(f'command failed with exit code {proc.returncode}: {cmd}')

## 2. Environment

In [ ]:
import numpy as np
import torch
import torchvision

print(f'python {sys.version.split()[0]} | torch {torch.__version__} | torchvision {torchvision.__version__} | numpy {np.__version__}')
assert torch.cuda.is_available(), 'No GPU found. Choose Settings > Accelerator > GPU T4 x2 and restart the session.'
print('GPU:', torch.cuda.get_device_name(0))

sh(f'{sys.executable} -m pip install -q easydict tqdm pyyaml scipy pillow tensorboard')

## 3. Code

Clones the repo (or copies your own copy from `CODE_INPUT`) and applies the one fix it needs on current NumPy.

In [ ]:
if not (REPO / 'train.py').exists():
    if CODE_INPUT:
        shutil.copytree(CODE_INPUT, REPO, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__', '*.pyc', '*.ipynb'))
    else:
        sh(f'git clone --depth 1 {REPO_URL} "{REPO}"')
assert (REPO / 'train.py').exists() and (REPO / 'models' / 'MTrans.py').exists(), f'{REPO} does not look like the MTrans repo'

# NumPy 1.24 removed np.int, which the IoU loss still uses at import time
source_file = REPO / 'loss' / 'utils' / 'min_enclosing_box.py'
source = source_file.read_text()
if 'astype(np.int)' in source:
    source_file.write_text(source.replace('astype(np.int)', 'astype(int)'))
    print('patched np.int in', source_file.relative_to(REPO))
assert 'np.int)' not in source_file.read_text()

## 4. IoU loss op

The box loss has to sort polygon vertices. The repo does this in a small CUDA extension. The cells below try to build it. If the build or its check fails, a pure-PyTorch version with the same contract is used instead. Either way `check_iou.py` must reproduce IoU values that are known in closed form before anything is trained.

In [ ]:
%%writefile /kaggle/working/MTrans/sort_vertices_fallback.py
"""Pure-PyTorch stand-in for the repo's compiled `sort_vertices` CUDA extension (loss/cuda_op).

Same contract as its sort_vertices_forward: for every polygon return the indices of its valid vertices in
angular order, then the first index again to close the polygon, then padding with an unused slot.
"""
import torch

MAX_NUM_VERT_IDX = 9
INTERSECTION_OFFSET = 8


def sort_vertices_forward(vertices, mask, num_valid):
    # vertices: (B, N, 24, 2), centred on each polygon's mean; mask: (B, N, 24) bool; num_valid: (B, N) int
    angle = torch.atan2(vertices[..., 1], vertices[..., 0])
    angle = torch.where(mask, angle, torch.full_like(angle, float('inf')))
    order = angle.argsort(dim=-1)[..., :MAX_NUM_VERT_IDX - 1]            # valid vertices first, by angle
    # an unused edge-intersection slot: its coordinates are zero, so it adds nothing to the shoelace area
    pad = (~mask[..., INTERSECTION_OFFSET:]).int().argmax(dim=-1, keepdim=True) + INTERSECTION_OFFSET
    count = num_valid.long().clamp(max=MAX_NUM_VERT_IDX - 1).unsqueeze(-1)
    slot = torch.arange(MAX_NUM_VERT_IDX, device=vertices.device).expand(*count.shape[:-1], -1)
    ring = torch.cat([order, order[..., :1]], dim=-1)                    # (B, N, 9)
    idx = torch.where(slot < count, ring, pad.expand_as(ring))
    idx = torch.where(slot == count, order[..., :1].expand_as(ring), idx)
    idx = torch.where(count < 3, pad.expand_as(ring), idx)               # fewer than 3 vertices: no polygon
    return idx.int()

In [ ]:
%%writefile /kaggle/working/MTrans/check_iou.py
"""Check the rotated-IoU loss against cases whose answer is known in closed form."""
import math

import torch

import sort_vertices
from loss import cal_diou_3d

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('sort_vertices loaded from', sort_vertices.__file__)

# boxes are (x, y, z, l, w, h, yaw)
octagon = 2 * (math.sqrt(2) - 1)      # area shared by a unit square and the same square turned 45 degrees
cases = [
    ('identical boxes',        [0, 0, 0, 4, 1.6, 1.5, 0.3], [0, 0, 0, 4, 1.6, 1.5, 0.3],     1.0),
    ('shifted by half length', [0, 0, 0, 4, 2, 1.5, 0],     [2, 0, 0, 4, 2, 1.5, 0],         1 / 3),
    ('shifted by half height', [0, 0, 0, 2, 2, 2, 0.5],     [0, 0, 1, 2, 2, 2, 0.5],         1 / 3),
    ('turned 45 degrees',      [0, 0, 0, 1, 1, 1, 0],       [0, 0, 0, 1, 1, 1, math.pi / 4], octagon / (2 - octagon)),
    ('typical prediction',     [0.3, -0.2, 0.1, 3.8, 1.7, 1.5, 0.4], [0, 0, 0, 4, 1.6, 1.5, 0.3], None),
    ('far apart',              [0, 0, 0, 1, 1, 1, 0],       [5, 5, 0, 1, 1, 1, 0.2],         0.0),
]
pred = torch.tensor([c[1] for c in cases], dtype=torch.float32, device=device).unsqueeze(1).requires_grad_()
target = torch.tensor([c[2] for c in cases], dtype=torch.float32, device=device).unsqueeze(1)
loss, iou3d, _ = cal_diou_3d(pred, target)
loss.mean().backward()

failed = False
for (name, _, _, expected), got in zip(cases, iou3d.flatten().tolist()):
    ok = 0.3 < got < 0.9 if expected is None else abs(got - expected) < 1e-3
    failed |= not ok
    wanted = 'between 0.3 and 0.9' if expected is None else f'{expected:.4f}'
    print(f'{"ok  " if ok else "FAIL"} {name:<24} IoU {got:.4f} (expected {wanted})')
assert not failed, 'the IoU op returned wrong values'
assert torch.isfinite(pred.grad).all(), 'the IoU loss produced a non-finite gradient'
print('IoU op check passed')

In [ ]:
fallback = REPO / 'sort_vertices.py'
fallback.unlink(missing_ok=True)      # a leftover fallback would shadow a freshly built extension


def iou_check():
    try:
        sh(f'{sys.executable} check_iou.py', cwd=REPO)
        return True
    except RuntimeError as err:
        print(err)
        return False


compiled = False
if USE_COMPILED_IOU:
    try:
        sh(f'{sys.executable} -m pip install -q --no-build-isolation .', cwd=REPO / 'loss' / 'cuda_op')
        compiled = iou_check()
    except RuntimeError as err:
        print(err)

if not compiled:
    shutil.copyfile(REPO / 'sort_vertices_fallback.py', fallback)
    assert iou_check(), 'The IoU check fails with the pure-PyTorch op too. Do not train until it passes.'

print('\nIoU op in use:', 'compiled CUDA extension' if compiled else 'pure-PyTorch fallback')

## 5. Data

Finds the KITTI folders among the attached inputs, links them into the layout the repo expects, downloads the standard train/val split, and restores the dataset cache and the latest checkpoint if an earlier output of this notebook is attached.

In [ ]:
NEEDED = ('image_2', 'velodyne', 'calib', 'label_2')
BIG_FOLDERS = set(NEEDED) | {'image_3', 'velodyne_reduced', 'gt_database', 'planes'}

kitti, old_caches, old_ckpts = {}, {}, []
for dirpath, dirnames, filenames in os.walk(INPUT_ROOT, followlinks=True):
    here = Path(dirpath)
    if here.name == 'training':
        for name in NEEDED:
            if name in dirnames:
                kitti.setdefault(name, here / name)
    for split in ('train', 'val'):
        if f'gt_set_{split}.pkl' in filenames:
            old_caches.setdefault(split, here / f'gt_set_{split}.pkl')
    if 'last.pt' in filenames and here.name == 'ckpt' and here.parent.name == EXPERIMENT:
        old_ckpts.append(here / 'last.pt')
    dirnames[:] = [d for d in dirnames if d not in BIG_FOLDERS]     # do not walk into the per-frame folders

missing = [name for name in NEEDED if name not in kitti]
assert not missing, (f'No training/{missing[0]} folder found under {INPUT_ROOT}. '
                     'Add a KITTI 3D object detection dataset with Add Input.')
for name, src in kitti.items():
    print(f'{name:>9}: {src}')

In [ ]:
# layout the repo expects: <DATA_ROOT>/training/{image_2,velodyne,calib,label_2} and <DATA_ROOT>/ImageSets/{train,val}.txt
(DATA_ROOT / 'training').mkdir(parents=True, exist_ok=True)
for name, src in kitti.items():
    link = DATA_ROOT / 'training' / name
    if link.is_symlink():
        link.unlink()
    link.symlink_to(src, target_is_directory=True)

# the standard 3712 / 3769 train / val split, as shipped with OpenPCDet
SPLIT_URL = 'https://raw.githubusercontent.com/open-mmlab/OpenPCDet/master/data/kitti/ImageSets/{}.txt'
(DATA_ROOT / 'ImageSets').mkdir(exist_ok=True)
available = {name: {Path(entry.name).stem for entry in os.scandir(src)} for name, src in kitti.items()}
for split, expected in (('train', 3712), ('val', 3769)):
    split_file = DATA_ROOT / 'ImageSets' / f'{split}.txt'
    if not split_file.exists():
        urllib.request.urlretrieve(SPLIT_URL.format(split), split_file)
    frames = split_file.read_text().split()
    assert len(frames) == expected, f'{split_file} has {len(frames)} frames, expected {expected}'
    for name in NEEDED:
        absent = sorted(set(frames) - available[name])
        assert not absent, f'{len(absent)} {split} frames are missing from {kitti[name]}, for example {absent[:3]}'
    print(f'{split}: {len(frames)} frames, all files present')

In [ ]:
# the object-level cache is written through this link into CACHE_DIR, so it is saved with the notebook output
(CACHE_DIR / 'gt_base').mkdir(parents=True, exist_ok=True)
link = DATA_ROOT / 'gt_base'
if link.is_symlink():
    link.unlink()
link.symlink_to(CACHE_DIR / 'gt_base', target_is_directory=True)
for split, src in old_caches.items():
    dst = CACHE_DIR / 'gt_base' / split / src.name
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, dst)
        print('reusing dataset cache from', src)


def epochs_done(last_pt):
    progress = Path(last_pt).parent / 'progress.json'
    return json.loads(progress.read_text())['epoch'] + 1 if progress.exists() else 0


# continue from an earlier session: take the checkpoint that got furthest
if not (CKPT_DIR / 'last.pt').exists():
    source = Path(RESUME_FROM) if RESUME_FROM else max(old_ckpts, key=epochs_done, default=None)
    if source is not None:
        CKPT_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, CKPT_DIR / 'last.pt')
        for name in ('best_model.pt', 'progress.json'):
            if (source.parent / name).exists():
                shutil.copyfile(source.parent / name, CKPT_DIR / name)
        print('copied checkpoint from', source)

if (CKPT_DIR / 'last.pt').exists():
    print(f'starting point: {epochs_done(CKPT_DIR / "last.pt")} epochs already done')
else:
    print('starting point: a fresh run')

## 6. Config

The repo's `configs/MTrans_kitti.yaml` with the paths and settings from section 1.

In [ ]:
import yaml

cfg = yaml.safe_load((REPO / 'configs' / 'MTrans_kitti.yaml').read_text())
cfg['data_root'] = str(DATA_ROOT)
cfg['experiment_name'] = EXPERIMENT
cfg['label_dir'] = str(LABEL_DIR)
cfg['TRAIN_CONFIG'].update(output_root=str(OUTPUT_ROOT), epochs=EPOCHS, epoches_per_eval=EVAL_EVERY, eval_begin=EVAL_BEGIN)
cfg['DATASET_CONFIG']['TRAIN_SET']['partial_frames'] = LABELLED_FRAMES
for loader in cfg['DATALOADER_CONFIG'].values():
    loader['num_workers'] = NUM_WORKERS
CFG_FILE.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(CFG_FILE.read_text())

## 7. Training driver

`train_kaggle.py` calls the repo's own `train_one_epoch` and `eval`. It replaces only the outer loop of `train.py`, which saves nothing before epoch 254 and resumes from the best checkpoint instead of the latest one.

In [ ]:
%%writefile /kaggle/working/MTrans/train_kaggle.py
"""Kaggle driver for MTrans.

Runs the same training as the repo's train.py (it calls train.train_one_epoch and train.eval) and adds
what a time-limited session needs:
  * a checkpoint after every epoch (ckpt/last.pt) and resume from it
  * a wall-clock budget, so the run stops cleanly before the session is killed
  * --build_only : build the dataset caches and exit
  * --smoke N    : one tiny epoch on N objects per set, exercising training, evaluation, checkpoints and labels
  * --eval_only  : evaluate a checkpoint, with --gen_label also write KITTI-format labels
"""
import argparse
import json
import os
import shutil
import time
from copy import deepcopy
from os import path

import torch
import yaml
from easydict import EasyDict
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm as _tqdm

import datasets.kitti_detection as kitti_detection
import train as T
from datasets.kitti_detection import KittiDetectionDataset
from datasets.kitti_loader import build_kitti_loader
from models.MTrans import MTrans
from utils.lr_scheduler import WarmupCosineAnnealing
from utils.stat_scores import HistoCounter, ScoreCounter


class QuietTqdm(_tqdm):
    """tqdm that redraws every 30 s rather than on every batch, so the saved log stays readable."""

    def __init__(self, *args, **kwargs):
        kwargs.setdefault('mininterval', 30)
        super().__init__(*args, **kwargs)

    def set_description(self, desc=None, refresh=True):
        super().set_description(desc, refresh=False)


T.tqdm = QuietTqdm
kitti_detection.tqdm = QuietTqdm


def save_last(ckpt_dir, epoch, total_epochs, model, optim, scheduler, best_score):
    os.makedirs(ckpt_dir, exist_ok=True)
    last_path = path.join(ckpt_dir, 'last.pt')
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_score': best_score,
    }, last_path + '.tmp')
    os.replace(last_path + '.tmp', last_path)       # a killed session never leaves a half-written checkpoint
    with open(path.join(ckpt_dir, 'progress.json'), 'w') as f:
        json.dump({'epoch': epoch, 'epochs': total_epochs, 'best_score': best_score}, f)
    return last_path


def load_checkpoint(file_path, model, optim=None, scheduler=None):
    ckpt = torch.load(file_path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    if optim is not None and 'optimizer_state_dict' in ckpt:
        optim.load_state_dict(ckpt['optimizer_state_dict'])
    if scheduler is not None and 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    return ckpt['epoch'], ckpt.get('best_score', -9999.0)


def build_datasets(cfg, smoke, with_train):
    dataset_cfg = cfg.DATASET_CONFIG
    training_set = unlabeled_set = None
    if with_train:
        training_set = KittiDetectionDataset(cfg.data_root, dataset_cfg.TRAIN_SET, nusc=None)
        if cfg.DATALOADER_CONFIG.TRAIN_LOADER.unsupervise_batch_size > 0:
            # the same split minus the labelled frames; these objects only feed the reconstruction loss
            unlabeled_cfg = deepcopy(dataset_cfg.TRAIN_SET)
            unlabeled_cfg.use_3d_label = False
            unlabeled_set = KittiDetectionDataset(cfg.data_root, unlabeled_cfg, nusc=None)
    validation_set = KittiDetectionDataset(cfg.data_root, dataset_cfg.VAL_SET, nusc=None)

    named = (('labelled train', training_set), ('unlabelled train', unlabeled_set),
             (f'evaluation ({dataset_cfg.VAL_SET.split})', validation_set))
    for name, dataset in named:
        if dataset is None:
            continue
        if smoke:
            dataset.objects = dataset.objects[:smoke]
        print(f'{name:>18}: {len(dataset)} objects')
        assert len(dataset) > 0, f'the {name} set is empty; delete the gt_base cache and build it again'
    return training_set, unlabeled_set, validation_set


def main(args):
    t_start = time.time()
    cfg = EasyDict(yaml.safe_load(open(args.cfg_file)))
    train_cfg, loader_cfg = cfg.TRAIN_CONFIG, cfg.DATALOADER_CONFIG
    if args.smoke:
        cfg.experiment_name = cfg.experiment_name + '_smoke'
        train_cfg.epochs, train_cfg.epoches_per_eval, train_cfg.eval_begin = 1, 1, 0
    if args.label_split:
        cfg.DATASET_CONFIG.VAL_SET.split = args.label_split
    output_path = path.join(train_cfg.output_root, cfg.experiment_name)
    ckpt_dir = path.join(output_path, 'ckpt')
    last_path, best_path = path.join(ckpt_dir, 'last.pt'), path.join(ckpt_dir, 'best_model.pt')
    if args.smoke:
        if path.isdir(output_path):
            shutil.rmtree(output_path)
        cfg.gen_label, cfg.label_dir = True, path.join(output_path, 'labels')

    T.freeze_random_seed(cfg.random_seed)
    training_set, unlabeled_set, validation_set = build_datasets(cfg, args.smoke, with_train=not args.eval_only)
    if args.build_only:
        print('Dataset caches are ready.')
        return

    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    print(f'torch {torch.__version__} on {device_name}')
    validation_loader = build_kitti_loader(validation_set, loader_cfg.VAL_LOADER)
    model = MTrans(cfg.MODEL_CONFIG).cuda()
    print('[Model Params]:', sum(p.numel() for p in model.parameters()))
    counter, histo_counter = ScoreCounter(), HistoCounter()

    if args.eval_only:
        checkpoint = args.checkpoint or next((p for p in (best_path, last_path) if path.exists(p)), None)
        assert checkpoint, f'--eval_only needs a checkpoint and none was found in {ckpt_dir}'
        epoch, _ = load_checkpoint(checkpoint, model)
        split = cfg.DATASET_CONFIG.VAL_SET.split
        print(f'Evaluating {checkpoint} (epoch {epoch}) on the {split} split')
        if args.gen_label:
            cfg.gen_label = True
            cfg.label_dir = path.join(cfg.label_dir, f'{split}_epoch{epoch}')
            if path.isdir(cfg.label_dir):
                shutil.rmtree(cfg.label_dir)        # train.eval appends to existing files, so start empty
        writer = SummaryWriter(log_dir=path.join(output_path, 'eval_only_tb'))
        score = float(T.eval(cfg, model, validation_loader, counter, histo_counter, epoch, writer))
        writer.close()
        print(f'Mean 3D IoU against ground truth: {score:.4f}')
        if args.gen_label:
            print(f'Labels written to {cfg.label_dir}')
        return

    training_loader = build_kitti_loader(training_set, loader_cfg.TRAIN_LOADER)
    unlabeled_loader = None
    if unlabeled_set is not None:
        unlabeled_loader_cfg = deepcopy(loader_cfg.TRAIN_LOADER)
        unlabeled_loader_cfg.batch_size = unlabeled_loader_cfg.unsupervise_batch_size
        unlabeled_loader = build_kitti_loader(unlabeled_set, unlabeled_loader_cfg)

    optim = getattr(torch.optim, train_cfg.optimizer)(lr=train_cfg.lr, params=model.parameters())
    scheduler = WarmupCosineAnnealing(optim, train_cfg.lr, train_cfg.warmup_rate,
                                      train_cfg.epochs * len(training_loader), eta_min=0)
    scheduler.step()

    start_epoch, best_score = 0, -9999.0
    checkpoint = None if args.smoke else (args.checkpoint or (last_path if path.exists(last_path) else None))
    if checkpoint:
        epoch_done, best_score = load_checkpoint(checkpoint, model, optim, scheduler)
        start_epoch = epoch_done + 1
        print(f'Resuming from {checkpoint}: {start_epoch} of {train_cfg.epochs} epochs done')
    if start_epoch >= train_cfg.epochs:
        print('All epochs are already done.')
        return

    writer = SummaryWriter(log_dir=path.join(output_path, cfg.experiment_name + '_tb'))
    budget_seconds = args.time_budget_hours * 3600
    epoch_seconds = []
    for epoch in range(start_epoch, train_cfg.epochs):
        t_epoch = time.time()
        T.train_one_epoch(cfg, model, training_loader, unlabeled_loader, optim, scheduler,
                          counter, histo_counter, epoch, writer)

        improved = False
        if (epoch + 1) % train_cfg.epoches_per_eval == 0 and epoch >= train_cfg.eval_begin:
            score = float(T.eval(cfg, model, validation_loader, counter, histo_counter, epoch, writer))
            improved = score > best_score
            best_score = max(best_score, score)
            print(f'[epoch {epoch}] validation mean 3D IoU {score:.4f} (best {best_score:.4f})', flush=True)

        epoch_seconds.append(time.time() - t_epoch)
        is_last = epoch + 1 == train_cfg.epochs
        # stop when the next epoch, allowing for an evaluation pass, might not fit in the budget
        out_of_time = not is_last and (time.time() - t_start) + 1.5 * max(epoch_seconds) > budget_seconds
        if improved or is_last or out_of_time or (epoch + 1) % args.save_every == 0:
            save_last(ckpt_dir, epoch, train_cfg.epochs, model, optim, scheduler, best_score)
            if improved:
                shutil.copyfile(last_path, best_path)
        writer.flush()
        mean_minutes = sum(epoch_seconds) / len(epoch_seconds) / 60
        print(f'[epoch {epoch}] took {epoch_seconds[-1] / 60:.1f} min; {train_cfg.epochs - epoch - 1} epochs left, '
              f'about {(train_cfg.epochs - epoch - 1) * mean_minutes / 60:.1f} h at this pace', flush=True)
        if out_of_time:
            print(f'Time budget reached after epoch {epoch}. The checkpoint is saved; run again to continue.')
            break
    writer.close()

    if args.smoke:
        load_checkpoint(last_path, model, optim, scheduler)
        label_files = sorted(os.listdir(cfg.label_dir))
        with open(path.join(cfg.label_dir, label_files[0])) as f:
            example = f.readline().strip()
        print(f'Smoke test passed: the checkpoint reloads and {len(label_files)} label files were written.')
        print('Example label line:', example)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--cfg_file', required=True, help='path to the YAML config')
    parser.add_argument('--checkpoint', default=None, help='checkpoint to start from (default: ckpt/last.pt if present)')
    parser.add_argument('--time_budget_hours', type=float, default=11.0, help='stop cleanly after this many hours')
    parser.add_argument('--save_every', type=int, default=1, help='save ckpt/last.pt every this many epochs')
    parser.add_argument('--build_only', action='store_true', help='build the dataset caches and exit')
    parser.add_argument('--smoke', type=int, default=0, metavar='N', help='one tiny epoch on N objects per set')
    parser.add_argument('--eval_only', action='store_true', help='evaluate a checkpoint and exit')
    parser.add_argument('--gen_label', action='store_true', help='with --eval_only: write KITTI-format labels')
    parser.add_argument('--label_split', choices=['train', 'val'], default=None, help='split to evaluate or label')
    main(parser.parse_args())

## 8. Dataset cache

The first run reads every frame once (about 7,500 images and point clouds) and stores one record per car. Later sessions reuse the cache from the attached output and skip this step.

In [ ]:
sh(f'{sys.executable} train_kaggle.py --cfg_file "{CFG_FILE}" --build_only', cwd=REPO)

## 9. Smoke test

One epoch on 64 objects, then evaluation, a checkpoint round trip and a label export. If anything in the pipeline is broken it fails here, not hours into the real run.

In [ ]:
if RUN_SMOKE_TEST:
    sh(f'{sys.executable} train_kaggle.py --cfg_file "{CFG_FILE}" --smoke 64', cwd=REPO)
    shutil.rmtree(OUTPUT_ROOT / f'{EXPERIMENT}_smoke')      # its checkpoint and labels were only a test

## 10. Train

The progress line shows running averages over the epoch: `L` total loss, `Seg` foreground IoU, `XYZ` reconstruction error in metres, `IoU` mean 3D IoU of the predicted boxes, `R` share of boxes with IoU above 0.7, `Dr` direction accuracy, `Cf` confidence error.

In [ ]:
if RUN_TRAINING:
    hours_left = TIME_BUDGET_HOURS - (time.time() - T0) / 3600
    assert hours_left > 0.25, 'The time budget is used up. Raise TIME_BUDGET_HOURS or start a new session.'
    sh(f'{sys.executable} train_kaggle.py --cfg_file "{CFG_FILE}" --time_budget_hours {hours_left:.2f}', cwd=REPO)

## 11. Progress and pseudo labels

Labels are exported once all epochs are done, from `best_model.pt`. The model takes KITTI's ground-truth 2D boxes as input, so it writes one 3D box for each car that has a 2D box in the label files.

In [ ]:
progress_file = CKPT_DIR / 'progress.json'
progress = json.loads(progress_file.read_text()) if progress_file.exists() else None
finished = bool(progress) and progress['epoch'] + 1 >= progress['epochs']
if progress:
    best = f"{progress['best_score']:.4f}" if progress['best_score'] >= 0 else 'not evaluated yet'
    print(f"{progress['epoch'] + 1} of {progress['epochs']} epochs done; best validation 3D IoU: {best}")
else:
    print('No checkpoint yet.')

if EXPORT_LABELS and finished:
    sh(f'{sys.executable} train_kaggle.py --cfg_file "{CFG_FILE}" --eval_only --gen_label --label_split {LABEL_SPLIT}', cwd=REPO)
    newest = max(LABEL_DIR.iterdir(), key=lambda p: p.stat().st_mtime)
    label_files = sorted(newest.glob('*.txt'))
    print(f'\n{len(label_files)} label files in {newest}. First file ({label_files[0].name}):')
    print(label_files[0].read_text())
elif EXPORT_LABELS:
    print('Labels are exported after the last epoch. Run this notebook again to continue training.')

## What is in the output

| Path under `/kaggle/working` | Content |
|---|---|
| `output/<EXPERIMENT>/ckpt/last.pt` | latest checkpoint: model, optimizer, scheduler; this is what a later session resumes from |
| `output/<EXPERIMENT>/ckpt/best_model.pt` | checkpoint with the best validation 3D IoU |
| `output/<EXPERIMENT>/ckpt/progress.json` | epochs done and best score |
| `output/<EXPERIMENT>/<EXPERIMENT>_tb/` | TensorBoard logs |
| `pseudo_label/<split>_epoch<N>/` | one KITTI label file per frame |
| `kitti_cache/` | dataset cache, reused by later sessions |
| `MTrans/` | the code that ran, including `train_kaggle.py` |